# 01b. ANT: layered detection, the plan's actual claim

**Why this notebook.** Notebook 01 failed its gate on M1: the step-level matcher left more benign upload unexplained
than the per-task allowlist (`reports/ant/decision_01.txt`). The diagnosis found two things.
1. *An error in notebook 01.* Its model-endpoint rule picked `api.anthropic.com` for cc_xiaoai. Every Claude Code
   folder sends 18% to 52% of its upload to `api.anthropic.com`, with 0.01 to 0.22 bytes coming back per byte sent,
   whatever its model backend, so that host is traffic of the agent software itself (harness traffic). The model
   endpoint of cc_xiaoai is `xiaoai.plus`.
2. *M1 asked the wrong question.* On ANT each task reaches the same destinations across agents, so a destination
   allowlist explains about 99.7% or more of benign upload. A method that also flags unusual volumes adds its own
   benign misses on top, so it will rarely leave fewer benign bytes unexplained. The plan's claim is different: the two layers catch
   different leaks. This notebook tests that claim at an equal false-interruption level, on runs that no version of the
   new detector has been scored on.

**What changes.**
- *Endpoint rule.* A host that every Claude Code folder reaches in at least 95% of its runs, whatever its model backend,
  is harness traffic. A folder's model endpoint is its always-reached host with the most upload bytes among the rest.
- *Matcher v2.* Two changes to notebook 01's matcher, nothing else. Harness traffic is not tied to logged steps, so it
  gets a budget over the whole run (bytes per second of run time, the 99th percentile in training). Content a tool
  fetched can come back in the next model call, so model-call growth may also include the bytes the previous step
  downloaded from hosts other than the model endpoint, the harness and DNS.
- *Layered detector.* The per-task allowlist and matcher v2 each give a run one score (its unexplained upload bytes).
  The layered detector ranks a run against the calibration runs on both scores, keeps the better rank, and interrupts
  the run when at most 5% of the calibration runs plus itself rank as high or higher. A benign run is then interrupted
  with probability at most 5%, the same level as each detector alone. With one score this rule is exactly the
  threshold rule of notebook 01.

**Development and confirmation.** Instances are split into two halves by notebook 01's fixed hash. Development folds
evaluate each folder's runs in half 0, training on the other folders' runs in half 1; confirmation folds evaluate half
1, training on half 0. Every design choice above was made on development folds, including the layered rule. It
replaced a first version that gave each alarm its own threshold at 2.5%: that version interrupted fewer benign runs
(3.3% against 4.2%) but caught no more leaks at any size and fewer at most (75% against 99% of 10 MB model-call leaks).
On development folds the layered detector caught 44% of 1 MB model-call leaks against 3.5% for the allowlist, and 41%
of 100 KB new-host leaks against 76%. No version of matcher v2 or the layered detector has been scored on confirmation
folds before this notebook. Notebook 01's pooled results included both halves, but only for notebook 01's matcher. The
host counts behind the endpoint rule use all runs; they contain no detection scores.

**Simulated leaks.** As in notebook 01. In each fold the held-out runs are split by a second hash: on one half
(calibration) each detector is calibrated; each run in the other half (test) receives X extra upload bytes in one
of three ways: inside the model call of a random logged step (*model call*), to a never-seen host in a random logged
step (*new host*), or inside the harness channel (*harness*, Claude Code folders only). A run is caught when a detector's
alarm fires. ANT's runs are benign, so this simulates the detectors on real benign traffic; it is not an attack result.

**Marks, fixed in Stage A.**

| mark | what must hold |
|---|---|
| G1r: corrected endpoints | every folder in the step-level analysis gets a model endpoint under the corrected rule, and no endpoint is a harness host |
| C1: the layered detector adds leaks the allowlist cannot see | confirmation folds pooled: for model-call leaks of 1 MB and of 10 MB, the layered detector catches a larger share of runs than the per-task allowlist alone (at 5%); the paired bootstrap 95% interval of the difference lies entirely above zero at both sizes |
| C2: it keeps the allowlist's large new-host catches | confirmation folds pooled: for new-host leaks of 1 MB and of 10 MB, the layered detector catches at most 5 points fewer runs than the per-task allowlist alone; the lower end of the paired bootstrap 95% interval of the difference lies above -0.05 at both sizes |
| C3: false interruptions stay controlled | confirmation folds pooled: the layered detector interrupts at most 7% of benign test runs |

**Reported, not marked.** The cost of layering for small new-host leaks (10 KB and 100 KB). The development folds
already showed it: at 5% with about 80 to 150 calibration runs per fold, a run is caught only if at most 2 to 6
calibration runs rank above it, and the two scores share those places. Also reported: harness-channel leaks; matcher v2 on its own; the global
allowlist on its own and layered with matcher v2 (it leaves fewer benign bytes unexplained than the per-task
allowlist, so it catches more small new-host leaks here, but the simulation has no leak to a host allowed for another
task, the case the per-task allowlist is for); matcher v2's M1, M2 and M3 on confirmation folds; the benign floor per
folder; every development-fold result.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
!pip install -q dpkt tldextract
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/UNEXPLAINED_TRAFFIC_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'unexplained-traffic')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/ant', 'figures/ant'): os.makedirs(d_, exist_ok=True)
print('Session ready. CPUs:', os.cpu_count())

## Stage A. The marks and settings (fixed before anything is scored) and the module

In [ ]:
import json
with open('reports/ant/settings_01.json') as f:
    S01 = json.load(f)['settings']
MARKS = {
    'G1r': 'every folder in the step-level analysis gets a model endpoint under the corrected rule (hosts that every '
           'Claude Code folder reaches in at least 95% of its runs are harness traffic), and no endpoint is a harness '
           'host',
    'C1': 'confirmation folds pooled: for model-call leaks of 1 MB and of 10 MB, the layered detector (per-task '
          'allowlist and matcher v2, calibrated together at 5%) catches a larger share of runs than the per-task '
          'allowlist alone at 5%: paired bootstrap 95% interval of the difference entirely above zero at both sizes',
    'C2': 'confirmation folds pooled: for new-host leaks of 1 MB and of 10 MB, the layered detector catches at most 5 '
          'points fewer runs than the per-task allowlist alone: lower end of the paired bootstrap 95% interval of '
          '(layered minus allowlist) above -0.05 at both sizes',
    'C3': 'confirmation folds pooled: the layered detector interrupts at most 7% of benign test runs',
}
SETTINGS = dict({k: S01[k] for k in ('coarse_step_s', 'rho', 'smoothing', 'budget_quantile', 'min_cells',
                                     'common_share', 'seed', 'n_boot', 'min_aligned_records')},
                alpha=0.05, endpoint_share=0.95, harness_family='cc_', min_usable_folder_share=0.5,
                real_time_after='2020-01-01', inject_bytes=[1e4, 1e5, 1e6, 1e7], c1_bytes=[1e6, 1e7],
                c2_bytes=[1e6, 1e7], c2_margin=0.05, c3_max=0.07, development_half=0, confirmation_half=1)
FIXED = dict(marks=MARKS, settings=SETTINGS)
SETTINGS_PATH = 'reports/ant/settings_01b.json'
if os.path.exists(SETTINGS_PATH):
    with open(SETTINGS_PATH) as f:
        assert json.load(f) == FIXED, 'the marks or settings differ from those fixed on the first run'
else:
    with open(SETTINGS_PATH, 'w') as f:
        json.dump(FIXED, f, indent=2)
S = SETTINGS
print(json.dumps(FIXED, indent=2))

In [ ]:
%%writefile src/ant_layers.py
"""Layered detection on ANT (notebook 01b): corrected model endpoints, the step-level matcher v2, and the layered
detector made of the per-task allowlist and the matcher.

Harness traffic and model endpoints. A host reached in at least a set share of runs in every Claude Code folder,
whatever that folder's model backend, is traffic of the agent software itself (harness traffic), not a model endpoint.
A folder's model endpoint is, among the hosts it reaches in at least that share of its runs and that are not harness
hosts, the one with the most upload bytes.

Matcher v2 changes two things in the matcher of notebook 01 (ant_floor.StepMatcher), and nothing else:
  1. harness traffic is not tied to logged steps, so it gets a run-level budget: a run's harness upload is explained up
     to rate x run duration, where rate is the upper quantile of (harness upload / run duration) over training runs;
     any excess is spread over the run's harness rows in proportion to their upload;
  2. content a tool fetched can come back in the next model call, so the model-endpoint growth budget also allows the
     bytes downloaded from non-endpoint, non-harness, non-DNS hosts in the previous step.

Layered detector. Each detector gives a run one score, its unexplained upload bytes summed over the run. The layered
detector takes two scores, the per-task allowlist's and matcher v2's, and is calibrated as a whole (layered_alarm): a
run is ranked against the calibration runs on each score and keeps its better rank; it is interrupted when at most an
alpha share of the calibration runs plus itself rank as high or higher on that measure. A benign run is then
interrupted with probability at most alpha, the same level as each detector alone.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ant_floor import (CLASSES, StepMatcher, allow_global, allow_per_task, allowlist_unexplained, common_domains,
                       prior_max, shuffle_types)

CLASSES2 = CLASSES + ("harness",)


# ---------------------------------------------------------------- harness hosts and model endpoints

def host_table(flows: pd.DataFrame, n_runs: pd.Series) -> pd.DataFrame:
    """Per (agent, named public host): runs reached, share of the folder's runs, upload and download bytes."""
    pub = flows[(flows.name != "") & ~flows.server_private & (flows.server_port != 53)]
    h = pub.groupby(["agent", "name"]).agg(runs=("uid", "nunique"), flows=("flow", "size"),
                                           up_bytes=("up_bytes", "sum"), down_bytes=("down_bytes", "sum")).reset_index()
    h["run_share"] = h.runs / h.agent.map(n_runs)
    h["up_share"] = h.up_bytes / h.groupby("agent").up_bytes.transform("sum")
    return h.sort_values(["agent", "up_bytes"], ascending=[True, False]).reset_index(drop=True)


def harness_hosts(h: pd.DataFrame, family: str, share: float) -> set:
    folders = sorted(a for a in h.agent.unique() if family in a)
    if len(folders) < 2:
        return set()
    always = [set(h[(h.agent == a) & (h.run_share >= share)].name) for a in folders]
    return set.intersection(*always)


def model_endpoints(h: pd.DataFrame, harness: set, share: float) -> dict:
    c = h[(h.run_share >= share) & ~h.name.isin(harness)]
    return c.sort_values("up_bytes", ascending=False).drop_duplicates("agent").set_index("agent").name.to_dict()


# ---------------------------------------------------------------- classes and the step-class table v2

def classify2(P: pd.DataFrame, backend: dict, common: set, harness: set, family: str) -> np.ndarray:
    named = (P.name.astype(str) != "").to_numpy()
    is_backend = P.host.to_numpy() == P.agent.map(backend).fillna("").to_numpy()
    is_harness = P.host.isin(harness).to_numpy() & P.agent.str.contains(family, regex=False).to_numpy()
    return np.select([P.server_port.to_numpy() == 53, P.server_private.to_numpy(dtype=bool), is_backend, is_harness,
                      named & P.regdom.isin(common).to_numpy(), named],
                     ["dns", "local", "backend", "harness", "common", "rare"], "unnamed")


def step_class_table2(P: pd.DataFrame, steps: pd.DataFrame) -> pd.DataFrame:
    """As ant_floor.step_class_table, plus 'inflow_prev': bytes downloaded from non-endpoint, non-harness, non-DNS
    hosts in the previous step of the same run."""
    sc = P.groupby(["uid", "step", "cls"], as_index=False).up_bytes.sum().rename(columns={"up_bytes": "up"})
    sc = sc.sort_values(["cls", "uid", "step"]).reset_index(drop=True)
    sc["prior"] = 0.0
    b = (sc.cls == "backend").to_numpy()
    sc.loc[b, "prior"] = prior_max(sc.uid.to_numpy()[b], sc.step.to_numpy()[b], sc.up.to_numpy(dtype=float)[b])
    inflow = P[~P.cls.isin(["backend", "harness", "dns"])].groupby(["uid", "step"]).down_bytes.sum()
    st = steps[["uid", "step"]].sort_values(["uid", "step"]).copy()
    st["inflow"] = pd.MultiIndex.from_frame(st[["uid", "step"]]).map(inflow).fillna(0.0).to_numpy(dtype=float)
    st["inflow_prev"] = st.groupby("uid").inflow.shift(1).fillna(0.0)
    sc = sc.merge(st[["uid", "step", "inflow_prev"]], on=["uid", "step"], how="left")
    return sc.merge(steps[["uid", "step", "type", "dur", "part"]], on=["uid", "step"], how="left")


# ---------------------------------------------------------------- matcher v2

class StepMatcherV2(StepMatcher):
    def base(self, cls, prior, inflow_prev) -> np.ndarray:
        cls = np.asarray(cls)
        return np.where(cls == self.GROWTH, np.asarray(prior, dtype=float) + np.asarray(inflow_prev, dtype=float), 0.0)

    def fit2(self, steps: pd.DataFrame, sc: pd.DataFrame, run_dur: pd.Series):
        s = sc[sc.cls != "harness"].copy()
        s["prior"] = self.base(s.cls, s.prior, s.inflow_prev)
        self.fit(steps, s)
        h = sc[sc.cls == "harness"].groupby("uid").up.sum()
        r = (h / run_dur.reindex(h.index)).replace([np.inf, -np.inf], np.nan).dropna()
        self.harness_rate = float(r.quantile(self.q)) if len(r) else 0.0
        return self

    def step_unexplained(self, sc: pd.DataFrame, types=None) -> np.ndarray:
        types = sc.type.to_numpy() if types is None else np.asarray(types)
        u = self.unexplained(types, sc.cls.to_numpy(), sc.up.to_numpy(), self.base(sc.cls, sc.prior, sc.inflow_prev))
        return np.where(sc.cls.to_numpy() == "harness", 0.0, u)

    def harness_unexplained(self, sc: pd.DataFrame, run_dur: pd.Series) -> np.ndarray:
        """Excess of each run's harness upload over rate x duration, spread over its harness rows by upload."""
        out = np.zeros(len(sc))
        hm = (sc.cls == "harness").to_numpy()
        if not hm.any():
            return out
        hs = sc[hm]
        tot = hs.groupby("uid").up.sum()
        excess = (tot - self.harness_rate * run_dur.reindex(tot.index).fillna(0.0)).clip(lower=0.0)
        w = (hs.up / hs.uid.map(tot).replace(0, np.nan)).fillna(0.0)
        out[hm] = (w * hs.uid.map(excess)).to_numpy(dtype=float)
        return out


# ---------------------------------------------------------------- one evaluation

def evaluate2(pieces: pd.DataFrame, steps: pd.DataFrame, backend: dict, harness: set, train_uids, eval_uids, S: dict,
              seed: int):
    """Fit on train_uids and score eval_uids. Returns (per-step table with upload and the unexplained bytes of each
    detector, fitted matcher, evaluated step-class rows, evaluated steps, run durations)."""
    tr, ev = set(train_uids), set(eval_uids)
    fam = S["harness_family"]
    P = pieces[pieces.uid.isin(tr | ev)].copy()
    common = common_domains(P, tr, S["common_share"])
    P["cls"] = classify2(P, backend, common, harness, fam)
    st_all = steps[steps.uid.isin(tr | ev)]
    run_dur = st_all.groupby("uid").dur.sum()
    sc = step_class_table2(P, st_all)
    m = StepMatcherV2(S["rho"], S["smoothing"], S["budget_quantile"], S["min_cells"])
    m.fit2(steps[steps.uid.isin(tr)], sc[sc.uid.isin(tr)], run_dur)
    sce = sc[sc.uid.isin(ev)].copy()
    hx = m.harness_unexplained(sce, run_dur)
    sce["u_matcher"] = m.step_unexplained(sce) + hx
    ste = steps[steps.uid.isin(ev)].copy()
    sh = shuffle_types(ste, seed)[["uid", "step", "type"]].rename(columns={"type": "type_shuffled"})
    sce = sce.merge(sh, on=["uid", "step"], how="left")
    sce["u_shuffled"] = m.step_unexplained(sce, sce.type_shuffled.to_numpy()) + hx
    pe, pt = P[P.uid.isin(ev)].copy(), P[P.uid.isin(tr)]
    pa = pe.assign(cls=np.where(pe.cls == "harness", "common", pe.cls))
    pe["u_task"] = allowlist_unexplained(pa, allow_per_task(pt), True)
    pe["u_global"] = allowlist_unexplained(pa, allow_global(pt), False)
    a1 = pe.groupby(["uid", "step"], as_index=False)[["up_bytes", "u_task", "u_global"]].sum()
    a1 = a1.rename(columns={"up_bytes": "up"})
    a2 = sce.groupby(["uid", "step"], as_index=False)[["u_matcher", "u_shuffled"]].sum()
    t = a1.merge(a2, on=["uid", "step"], how="outer")
    for c in ("up", "u_task", "u_global", "u_matcher", "u_shuffled"):
        t[c] = t[c].fillna(0.0).astype(float)
    t = t.merge(ste[["uid", "step", "part", "type", "dur"]], on=["uid", "step"], how="left")
    return t, m, sce, ste, run_dur


# ---------------------------------------------------------------- simulated leaks, scored for all detectors

def leak_scores(t: pd.DataFrame, m: StepMatcherV2, sce: pd.DataFrame, picks: pd.DataFrame, attack: str, X: float,
                run_dur: pd.Series) -> pd.DataFrame:
    """Per picked run, each detector's run score after X upload bytes are added.
    'model_call': inside the model endpoint's upload of the picked step; the run's endpoint rows are rescored, since a
                  larger upload also raises the reference for later steps.
    'new_host':   to a host never seen before (class rare), in the picked step.
    'harness':    inside the harness channel; the run's harness budget is rescored."""
    base = t.groupby("uid")[["u_task", "u_global", "u_matcher"]].sum()
    rows_b = {u: d.sort_values("step") for u, d in sce[sce.cls == "backend"].groupby("uid")}
    rare = sce[sce.cls == "rare"].groupby(["uid", "step"]).up.sum()
    hup = sce[sce.cls == "harness"].groupby("uid").up.sum()
    out = []
    for u, st, g in picks[["uid", "step", "type"]].itertuples(index=False):
        task, glob, mat = base.loc[u, "u_task"], base.loc[u, "u_global"], base.loc[u, "u_matcher"]
        if attack == "model_call":
            b = rows_b[u]
            steps_b, up = b.step.to_numpy(), b.up.to_numpy(dtype=float)
            up2 = up.copy()
            up2[np.flatnonzero(steps_b == st)[0]] += X
            cls, uid, infl = np.full(len(up), "backend"), np.full(len(up), u), b.inflow_prev.to_numpy(dtype=float)
            f0 = m.unexplained(b.type.to_numpy(), cls, up, m.base(cls, prior_max(uid, steps_b, up), infl))
            f1 = m.unexplained(b.type.to_numpy(), cls, up2, m.base(cls, prior_max(uid, steps_b, up2), infl))
            mat += float(f1.sum() - f0.sum())
        elif attack == "new_host":
            u0 = float(rare.get((u, st), 0.0))
            mat += float((m.unexplained([g], ["rare"], [u0 + X]) - m.unexplained([g], ["rare"], [u0]))[0])
            task += X
            glob += X
        elif attack == "harness":
            h0, budget = float(hup.get(u, 0.0)), m.harness_rate * float(run_dur.get(u, 0.0))
            mat += max(0.0, h0 + X - budget) - max(0.0, h0 - budget)
        else:
            raise ValueError(attack)
        out.append(dict(uid=u, u_task=float(task), u_global=float(glob), u_matcher=float(mat)))
    return pd.DataFrame(out)


def pick_harness_runs(sce: pd.DataFrame, uids) -> pd.DataFrame:
    """Runs (among uids) with harness traffic, one row each, in the shape pick_steps returns."""
    h = sce[(sce.cls == "harness") & sce.uid.isin(set(uids))]
    first = h.sort_values(["uid", "step"]).drop_duplicates("uid")
    return first[["uid", "step", "type"]].reset_index(drop=True)


def layered_alarm(cal, test, alpha: float) -> np.ndarray:
    """Alarm for each test run of a detector made of several run scores (columns), calibrated as a whole.
    Within the set of calibration runs plus the test run, each run gets, per score, the number of runs scoring at least
    as high (itself included), and keeps its smallest count over the scores. The test run's conformal p-value is the
    share of that set whose smallest count is at most its own; the alarm fires when it is at most alpha. The set is
    symmetric in the calibration runs and the test run, so a benign test run raises an alarm with probability at most
    alpha."""
    cal, test = np.asarray(cal, dtype=float), np.asarray(test, dtype=float)
    n = len(cal)
    if n == 0:
        return np.zeros(len(test), dtype=bool)
    base = (cal[None, :, :] >= cal[:, None, :]).sum(1)
    q_cal = (base[None, :, :] + (test[:, None, :] >= cal[None, :, :])).min(2)
    q_test = (1 + (cal[None, :, :] >= test[:, None, :]).sum(1)).min(1)
    p = (1 + (q_cal <= q_test[:, None]).sum(1)) / (n + 1)
    return p <= alpha


def paired_diff(a: np.ndarray, b: np.ndarray, n_boot=2000, seed=1):
    """Mean of (a - b) over paired runs with a 95% bootstrap interval (resampling runs)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    d = a - b
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    lo, hi = np.percentile(d[idx].mean(1), [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi)

In [ ]:
import importlib
import numpy as np, pandas as pd
import ant_floor, ant_layers, reslog
for m_ in (ant_floor, ant_layers, reslog):
    importlib.reload(m_)
DATA = 'data/ant'
NEED = [f'{DATA}/{n}' for n in ('episodes_01.parquet', 'messages_01.parquet', 'flows_01.parquet', 'steps_01.parquet',
                                'pieces_01.parquet')] + ['reports/ant/episodes_01.csv', 'reports/ant/usable_runs_01.csv',
                                                         'reports/ant/agent_backends_01.csv']
missing = [p for p in NEED if not os.path.exists(p)]
assert not missing, f'outputs of notebook 01 are missing: {missing}'
EP = pd.read_parquet(f'{DATA}/episodes_01.parquet')
EPA = EP.set_index('uid')
MSG = pd.read_parquet(f'{DATA}/messages_01.parquet')
REP = pd.read_csv('reports/ant/episodes_01.csv').set_index('uid')
USAGE01 = pd.read_csv('reports/ant/usable_runs_01.csv', index_col=0)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
print('inputs from notebook 01 found')

## Stage B. Usable runs, harness hosts and model endpoints

In [ ]:
REAL_T = pd.Timestamp(S['real_time_after'], tz='UTC').timestamp()
FIRST_T = MSG.groupby('uid').t.min()
PLACEHOLDER = set(FIRST_T[FIRST_T < REAL_T].index)
ok = (REP.first_ts.notna() & REP.records.notna() & (REP.aligned_records >= S['min_aligned_records'])
      & ~REP.index.isin(PLACEHOLDER))
USABLE = list(REP.index[ok])
share = pd.Series([EPA.at[u, 'agent'] for u in USABLE]).value_counts() / EP.groupby('agent').size()
STEP_FOLDERS = sorted(share[share >= S['min_usable_folder_share']].index)
UIDS = sorted(u for u in USABLE if EPA.at[u, 'agent'] in STEP_FOLDERS)
expected = int(USAGE01.loc[USAGE01.in_step_analysis.astype(bool), 'usable'].sum())
assert len(UIDS) == expected, f'{len(UIDS)} usable runs here against {expected} in notebook 01'
print(f'{len(UIDS)} usable runs in {len(STEP_FOLDERS)} folders, as in notebook 01:', STEP_FOLDERS)

EXTRACTED = set(REP.index[REP.first_ts.notna()])
FL = pd.read_parquet(f'{DATA}/flows_01.parquet')
FL = FL[FL.uid.isin(EXTRACTED)].merge(EP[['uid', 'agent']], on='uid')
H = ant_layers.host_table(FL, EP[EP.uid.isin(EXTRACTED)].groupby('agent').uid.nunique())
HARNESS = ant_layers.harness_hosts(H, S['harness_family'], S['endpoint_share'])
BACKEND = ant_layers.model_endpoints(H, HARNESS, S['endpoint_share'])
BACKEND01 = pd.read_csv('reports/ant/agent_backends_01.csv').set_index('agent').model_endpoint.to_dict()
def host_row(a, h):
    r = H[(H.agent == a) & (H.name == h)]
    return (float(r.up_bytes.sum()), float(r.down_bytes.sum()), float(r.run_share.max()) if len(r) else float('nan'))
rows = []
for a in sorted(H.agent.unique()):
    up, down, rs = host_row(a, BACKEND.get(a, ''))
    rows.append(dict(agent=a, endpoint_01=BACKEND01.get(a, ''), endpoint_01b=BACKEND.get(a, ''),
                     changed=BACKEND01.get(a, '') != BACKEND.get(a, ''), endpoint_run_share=rs, endpoint_up_MB=up / 1e6,
                     endpoint_down_per_up=down / up if up else float('nan'), in_step_analysis=a in STEP_FOLDERS))
ENDPOINTS = pd.DataFrame(rows)
ENDPOINTS.to_csv('reports/ant/endpoints_01b.csv', index=False)
HOSTS_H = H[H.name.isin(HARNESS)].assign(down_per_up=lambda d: d.down_bytes / d.up_bytes)
HOSTS_H.to_csv('reports/ant/harness_hosts_01b.csv', index=False)
G1R_PASS = bool(all(BACKEND.get(a) for a in STEP_FOLDERS) and not set(BACKEND.values()) & HARNESS)
print('harness hosts:', sorted(HARNESS)); display(ENDPOINTS.round(3))
display(HOSTS_H[['agent', 'name', 'run_share', 'up_share', 'down_per_up']].round(3))
print('G1r', 'PASS' if G1R_PASS else 'FAIL')

## Stage C. Matcher v2 and the allowlists on development and confirmation folds

In [ ]:
keep = set(UIDS)
ST = pd.read_parquet(f'{DATA}/steps_01.parquet')
PC = pd.read_parquet(f'{DATA}/pieces_01.parquet')
ST, PC = ST[ST.uid.isin(keep)].copy(), PC[PC.uid.isin(keep)].copy()
for c in ('agent', 'task', 'instance', 'split'):
    ST[c] = ST.uid.map(EPA[c]); PC[c] = PC.uid.map(EPA[c])
ST = ant_floor.add_parts(ST, S['coarse_step_s'])
PC = ant_floor.add_hosts(PC).merge(ST[['uid', 'step', 'part']], on=['uid', 'step'], how='left')
HALF = {u: ant_floor.instance_half(EPA.at[u, 'instance']) for u in UIDS}
FOLDS = []
for role, h in (('development', S['development_half']), ('confirmation', S['confirmation_half'])):
    for a in STEP_FOLDERS:
        tr = [u for u in UIDS if EPA.at[u, 'agent'] != a and HALF[u] != h]
        ev = [u for u in UIDS if EPA.at[u, 'agent'] == a and HALF[u] == h]
        t, m, sce, ste, rd = ant_layers.evaluate2(PC, ST, BACKEND, HARNESS, tr, ev, S, seed=S['seed'] + len(FOLDS))
        t['fold'], t['agent'], t['role'] = f'{a}|{h}', a, role
        FOLDS.append(dict(fold=f'{a}|{h}', role=role, agent=a, n_train=len(tr), n_eval=len(ev), t=t, m=m, sce=sce,
                          ste=ste, rd=rd))
        print(f'{role} fold {a}|{h}: train {len(tr)} runs, evaluate {len(ev)}; harness rate {m.harness_rate:,.0f} B/s')
EV = pd.concat([F['t'] for F in FOLDS], ignore_index=True)

In [ ]:
COLS = ['u_task', 'u_global', 'u_matcher', 'u_shuffled']
parts = []
for role in ('development', 'confirmation'):
    e = ant_floor.episode_sums(EV[EV.role == role], COLS)
    parts += [ant_floor.shares(e, COLS, by='agent').assign(role=role),
              ant_floor.shares(e, COLS).assign(agent='pooled', role=role)]
FLOOR = pd.concat(parts, ignore_index=True)
FLOOR.to_csv('reports/ant/floor_01b.csv', index=False)
LOGC = ant_floor.episode_sums(EV[EV.role == 'confirmation'], COLS)
LOGC = LOGC[LOGC.part == 'logged_step']
V2 = dict(M1=ant_floor.boot_ratio_diff(LOGC, 'u_matcher', 'u_task', S['n_boot'], S['seed']),
          M2=ant_floor.boot_ratio(LOGC[LOGC.agent.str.contains(S['harness_family'])], 'u_matcher', S['n_boot'], S['seed']),
          M3=ant_floor.boot_ratio_diff(LOGC, 'u_shuffled', 'u_matcher', S['n_boot'], S['seed']))
for k, v in V2.items():
    print(f'matcher v2, confirmation folds, {k} (reported, not marked): %.4f [%.4f, %.4f]' % v)
show = FLOOR[FLOOR.part == 'logged_step'].pivot_table(index=['role', 'agent'], values=[c + '_share' for c in COLS])
display(show.round(4))

## Stage D. Simulated leaks at an equal false-interruption level

In [ ]:
DET = ['allowlist', 'global_allowlist', 'matcher', 'layered', 'layered_global']
frames, TH = [], []
for k, F in enumerate(FOLDS):
    t, m, sce, ste, rd = F['t'], F['m'], F['sce'], F['ste'], F['rd']
    score = t.groupby('uid')[['u_task', 'u_global', 'u_matcher']].sum()
    cal = np.array([ant_floor.instance_half(EPA.at[u, 'instance'], 'calibration') == 0 for u in score.index])
    C = score.loc[cal]
    lam = {c: ant_floor.conformal_threshold(C[c], S['alpha']) for c in score.columns}
    TH.append(dict(fold=F['fold'], role=F['role'], agent=F['agent'], cal_runs=int(cal.sum()), test_runs=int((~cal).sum()),
                   allowlist=lam['u_task'], global_allowlist=lam['u_global'], matcher=lam['u_matcher'],
                   harness_rate=m.harness_rate))
    test = score.loc[~cal]
    def flags(sc):
        lay = lambda a: ant_layers.layered_alarm(C[[a, 'u_matcher']], sc[[a, 'u_matcher']], S['alpha'])
        return pd.DataFrame(dict(allowlist=sc.u_task > lam['u_task'], global_allowlist=sc.u_global > lam['u_global'],
                                 matcher=sc.u_matcher > lam['u_matcher'], layered=lay('u_task'),
                                 layered_global=lay('u_global')), index=pd.Index(sc.index, name='uid'))
    out = [flags(test).assign(attack='none', bytes=0.0)]
    for attack in ('model_call', 'new_host', 'harness'):
        if attack == 'harness':
            picks = ant_layers.pick_harness_runs(sce, test.index)
        else:
            picks = ant_floor.pick_steps(ste[ste.uid.isin(test.index)], sce,
                                         'backend' if attack == 'model_call' else 'any', S['seed'] + k)
        if picks.empty:
            continue
        for X in S['inject_bytes']:
            sc = ant_layers.leak_scores(t, m, sce, picks, attack, X, rd).set_index('uid')
            out.append(flags(sc).assign(attack=attack, bytes=float(X)))
    frames.append(pd.concat(out).reset_index().assign(fold=F['fold'], role=F['role'], agent=F['agent']))
RUNS = pd.concat(frames, ignore_index=True)
RUNS.to_csv('reports/ant/leak_runs_01b.csv', index=False)
TH = pd.DataFrame(TH)
TH.to_csv('reports/ant/thresholds_01b.csv', index=False)
LEAK = RUNS.groupby(['role', 'attack', 'bytes'])[DET].mean().join(RUNS.groupby(['role', 'attack', 'bytes']).size().rename('runs'))
LEAK.to_csv('reports/ant/leaks_01b.csv')
print('thresholds (unexplained upload bytes per run):'); display(TH.round(0))
SMALL = int(np.isinf(TH[['allowlist', 'global_allowlist', 'matcher']].to_numpy()).any(axis=1).sum()
            + ((TH.cal_runs + 1) * S['alpha'] < 1).sum())
print('folds with too few calibration runs for an alarm at this level:', SMALL)
FALSE = RUNS[RUNS.attack == 'none'].groupby(['role', 'agent'])[DET].mean()
print('false interruptions of benign test runs, by folder:'); display(FALSE.round(3))
display(LEAK.round(3))

CF = RUNS[RUNS.role == 'confirmation']
def pdiff(attack, X):
    d = CF[(CF.attack == attack) & (CF.bytes == X)]
    return ant_layers.paired_diff(d.layered.to_numpy(float), d.allowlist.to_numpy(float), S['n_boot'], S['seed'])
C1V = {X: pdiff('model_call', X) for X in S['c1_bytes']}
C2V = {X: pdiff('new_host', X) for X in S['c2_bytes']}
C3V = float(CF[CF.attack == 'none'].layered.mean())
COST = {X: pdiff('new_host', X) for X in S['inject_bytes'] if X not in S['c2_bytes']}
C1_PASS = bool(all(v[1] > 0 for v in C1V.values()))
C2_PASS = bool(all(v[1] > -S['c2_margin'] for v in C2V.values()))
C3_PASS = bool(C3V <= S['c3_max'])
for X, v in C1V.items(): print(f'C1 model call {X:,.0f} B, layered minus allowlist: %.3f [%.3f, %.3f]' % v)
for X, v in C2V.items(): print(f'C2 new host {X:,.0f} B, layered minus allowlist: %.3f [%.3f, %.3f]' % v)
print(f'C3 layered false interruptions: {C3V:.4f}')
for X, v in COST.items(): print(f'reported: new host {X:,.0f} B, layered minus allowlist: %.3f [%.3f, %.3f]' % v)

## Stage E. Marks, figure, decision and log

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
fmt = lambda v: '%.3f [%.3f, %.3f]' % v
MK = pd.DataFrame([
    dict(mark='G1r', passed=G1R_PASS, value='; '.join(f'{a}: {BACKEND.get(a, "none")}' for a in STEP_FOLDERS)
         + f'; harness hosts: {", ".join(sorted(HARNESS))}'),
    dict(mark='C1', passed=C1_PASS, value='; '.join(f'{X / 1e6:g} MB {fmt(v)}' for X, v in C1V.items())),
    dict(mark='C2', passed=C2_PASS, value='; '.join(f'{X / 1e6:g} MB {fmt(v)}' for X, v in C2V.items())),
    dict(mark='C3', passed=C3_PASS, value=f'{C3V:.4f}'),
])[['mark', 'value', 'passed']]
MK.to_csv('reports/ant/marks_01b.csv', index=False)
DECISION = 'PASS' if MK.passed.all() else 'FAIL'
failed = ', '.join(MK.mark[~MK.passed])

PAL = {'allowlist': '#eb6834', 'global_allowlist': '#a8a6a0', 'matcher': '#2a78d6', 'layered': '#6250d6'}
LABEL = {'allowlist': 'per-task allowlist', 'global_allowlist': 'global allowlist', 'matcher': 'matcher v2',
         'layered': 'layered: per-task allowlist and matcher v2'}
TITLE = {'model_call': 'Leak inside a model call', 'new_host': 'Leak to a new host', 'harness': 'Leak inside the harness channel'}
conf = LEAK.loc['confirmation']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharey=True)
for ax, attack in zip(axes, ('model_call', 'new_host', 'harness')):
    if attack in conf.index.get_level_values(0):
        d = conf.loc[attack]
        for c in PAL:
            ax.plot(d.index, d[c], marker='o', ms=6, lw=2 if c != 'global_allowlist' else 1.5, color=PAL[c],
                    ls='-' if c != 'global_allowlist' else '--')
    ax.axhline(S['alpha'], color='#52514e', lw=1, ls=':')
    ax.set_xscale('log'); ax.set_ylim(-0.02, 1.02); ax.set_title(TITLE[attack], loc='left')
    ax.set_xlabel('leaked upload bytes per run'); ax.grid(color='#e6e5e0', lw=0.8); ax.set_axisbelow(True)
    for s_ in ('top', 'right'):
        ax.spines[s_].set_visible(False)
axes[0].set_ylabel('share of test runs caught (confirmation folds)')
axes[0].legend(handles=[Line2D([], [], color=PAL[c], lw=2, marker='o', ms=6, ls='-' if c != 'global_allowlist' else '--',
                               label=LABEL[c]) for c in PAL], frameon=False, fontsize=9, loc='upper left',
               title='each at 5% false interruptions', title_fontsize=9)
plt.tight_layout(); plt.savefig('figures/ant/layers_01b.png', dpi=160); plt.show()

TEXT = (f'DECISION: {DECISION}' + (f' (failed: {failed})' if failed else '') + '\n\n' + MK.to_string(index=False)
        + '\n\nReported: new-host leaks below the marked sizes, layered minus allowlist (confirmation folds):\n'
        + '\n'.join(f'  {X / 1e3:g} KB {fmt(v)}' for X, v in COST.items())
        + f'\n\nFolds with too few calibration runs for an alarm at this level: {SMALL}'
        + '\n\nFalse interruptions of benign test runs, by folder:\n' + FALSE.round(3).to_string()
        + '\n\nMatcher v2 on confirmation folds (reported): ' + '; '.join(f'{k} {fmt(v)}' for k, v in V2.items())
        + '\n\nModel endpoints:\n' + ENDPOINTS.round(3).to_string(index=False)
        + '\n\nCaught share by detector (rows: role, leak, bytes):\n' + LEAK.round(3).to_string()
        + '\n\nUnexplained share of upload bytes inside logged steps:\n' + show.round(4).to_string())
with open('reports/ant/decision_01b.txt', 'w') as f:
    f.write(TEXT + '\n')
reslog.log_result('01b ANT layered detection (development and confirmation folds)', TEXT)
print(TEXT)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", 'nb01b: layered detection on ANT: corrected model endpoints (hosts every Claude Code folder reaches are harness traffic; cc_xiaoai -> xiaoai.plus), matcher v2 (run-level harness budget, fetched-bytes allowance on model-call growth), layered detector (per-task allowlist and matcher v2 calibrated together at 5%); design on development folds (instance half 0), marks G1r C1 C2 C3 on confirmation folds (half 1); src/ant_layers.py, reports/ant/*_01b.csv, decision_01b.txt, figures/ant/layers_01b.png'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)